## Background

Taiwan (officially the Republic of China) maintains formal diplomatic recognition with only a small number of countries worldwide. As of 2024, only **12 countries** officially recognize Taiwan as a sovereign state, down from over 20 in the early 2000s. The rest of the world, including major economies, recognizes the **People's Republic of China (PRC)** under the "One China" policy.

Over the past two decades, several countries have **switched their diplomatic recognition from Taiwan to China**, often citing economic incentives, foreign aid packages, or geopolitical pressure from Beijing. Understanding which countries switched, when, and what their trade relationships looked like before and after is the central focus of this project.


## Research Question

> **Which of Taiwan's current diplomatic allies have the strongest and most stable trade relationships with Taiwan, and how did trade change for countries that switched recognition to China?**


## Countries in This Study

**Countries that switched recognition from Taiwan to China (Switchers):**

| Country | Switch Year | Region |
|---|---|---|
| Costa Rica | 2007 | Latin America |
| Panama | 2017 | Latin America |
| El Salvador | 2018 | Latin America |
| Nicaragua | 2021 | Latin America |
| Honduras | 2023 | Latin America |

**Current Taiwan Allies (still recognize Taiwan as of 2024):**
Belize, Eswatini, Guatemala, Haiti, Holy See, Marshall Islands, Palau, Paraguay, Saint Kitts and Nevis, Saint Lucia, Saint Vincent and the Grenadines, Tuvalu

## Data Sources

- **Taiwan Trade Data:** Taiwan Ministry of Finance — bilateral exports and imports (USD thousands), 2003–2024
- **China Trade Data:** World Bank World Integrated Trade Solution (WITS) — bilateral exports and imports between China and partner countries (USD thousands), 2003–2023
- **GDP Data:** World Bank World Development Indicators
- **Regime Type:** V-Dem Dataset v16 (electoral democracy score)
- **Recognition Panel:** Original dataset tracking switch years


*Use the interactive charts below to explore the data. Hover over any line or bar to see exact values. Use the legend to toggle countries on and off.*

In [1]:
# ── Setup ─────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

pio.renderers.default = 'plotly_mimetype+notebook_connected'

# ── Color palette ──────────────────────────────────────────────────────
SWITCHER_COLOR = '#E05C5C'   # red  — countries that switched
ALLY_COLOR     = '#4A90D9'   # blue — current Taiwan allies
ACCENT         = '#2E4057'   # dark navy
HIGHLIGHT      = '#E8A838'   # gold

# Country-specific color map for switchers
SWITCHER_COLORS = {
    'Costa Rica':   '#E05C5C',
    'Panama':       '#E8A838',
    'El Salvador':  '#9B59B6',
    'Nicaragua':    '#E67E22',
    'Honduras':     '#1ABC9C',
}

ALLY_COLORS = {
    'Guatemala':    '#4A90D9',
    'Paraguay':     '#2E75B6',
    'Belize':       '#5BAD6F',
    'Haiti':        '#34495E',
}

print('✓ Libraries loaded successfully')

✓ Libraries loaded successfully


In [2]:
# ── Load Taiwan trade data ─────────────────────────────────────────────
tw_path = "/Users/jasminehernandez/Desktop/Taiwan Research Project/Capstone reasearch Data/Taiwan Exports and Imports Data.csv"

tw = pd.read_csv(tw_path)
tw = tw.rename(columns={
    "Imports/Exports": "Flow",
    "Time": "Year",
    "Country(Area)": "Partner",
    "Value(USD$ 1000)": "Value"
})
tw["Year"] = pd.to_numeric(tw["Year"], errors="coerce")
tw["Value"] = pd.to_numeric(tw["Value"], errors="coerce")
tw = tw.dropna(subset=["Year", "Value"])
tw["Year"] = tw["Year"].astype(int)
tw = tw[tw["Year"].between(2003, 2024)]

# ── Define country groups ──────────────────────────────────────────────
switch_years = {
    "Costa Rica":   2007,
    "Panama":       2017,
    "El Salvador":  2018,
    "Nicaragua":    2021,
    "Honduras":     2023,
}

current_allies = ["Guatemala", "Paraguay", "Belize", "Haiti",
                  "Marshall Islands", "Palau", "Tuvalu",
                  "Saint Kitts and Nevis", "Saint Lucia",
                  "Saint Vincent and the Grenadines"]

all_countries = list(switch_years.keys()) + current_allies

# ── Build pivot table (Exports + Imports → Total Trade) ────────────────
tw_focus = tw[tw["Partner"].isin(all_countries)].copy()

tw_pivot = tw_focus.pivot_table(
    index=["Year", "Partner"],
    columns="Flow",
    values="Value",
    aggfunc="sum"
).reset_index()
tw_pivot.columns.name = None
tw_pivot = tw_pivot.fillna(0)
tw_pivot["Total Trade"] = tw_pivot.get("Exports", 0) + tw_pivot.get("Imports", 0)
tw_pivot["Exports"]     = tw_pivot.get("Exports", 0)
tw_pivot["Imports"]     = tw_pivot.get("Imports", 0)

# ── Add group labels and switch info ──────────────────────────────────
tw_pivot["Group"] = tw_pivot["Partner"].apply(
    lambda x: "Switcher" if x in switch_years else "Current Ally"
)
tw_pivot["Switch Year"] = tw_pivot["Partner"].map(switch_years)
tw_pivot["Period"] = tw_pivot.apply(
    lambda r: "Before Switch" if (r["Partner"] in switch_years and r["Year"] < switch_years[r["Partner"]])
    else ("After Switch" if r["Partner"] in switch_years else "Current Ally"),
    axis=1
)

# Growth rate
tw_pivot = tw_pivot.sort_values(["Partner", "Year"])
tw_pivot["Trade_Growth_Pct"] = (
    tw_pivot.groupby("Partner")["Total Trade"].pct_change() * 100
)

print(f"✓ Data loaded: {len(tw_pivot)} rows across {tw_pivot['Partner'].nunique()} countries")
tw_pivot.head()

✓ Data loaded: 329 rows across 15 countries


,Year,Partner,Exports,Imports,Total Trade,Group,Switch Year,Period,Trade_Growth_Pct
0,2003,Belize,6032.46645,658.70153,6691.16798,Current Ally,NaN,Current Ally,NaN
14,2004,Belize,6237.70310,2011.36655,8249.06965,Current Ally,NaN,Current Ally,23.282956
29,2005,Belize,8101.08930,2770.40798,10871.49728,Current Ally,NaN,Current Ally,31.790587
44,2006,Belize,4094.68304,2719.37803,6814.06107,Current Ally,NaN,Current Ally,-37.321779
59,2007,Belize,6050.05207,1833.69105,7883.74312,Current Ally,NaN,Current Ally,15.698158


## Taiwan Trade Over Time

The charts in this section show how Taiwan's total bilateral trade (exports + imports combined) changed over time for both **switcher countries** and **current Taiwan allies**.

The vertical dashed lines on the switcher charts mark the year each country switched diplomatic recognition from Taiwan to China. Watching what happens to trade volumes around those switch years reveals whether switching had an economic impact on the relationship.

**How to use these charts:** Hover over any line to see the exact trade value for that year. Click country names in the legend to show or hide specific countries.

In [3]:
# ── Chart 1: Taiwan Total Trade — Switcher Countries ──────────────────
switcher_data = tw_pivot[tw_pivot["Group"] == "Switcher"].copy()

fig = go.Figure()

for country, sy in switch_years.items():
    df_c = switcher_data[switcher_data["Partner"] == country]
    color = SWITCHER_COLORS.get(country, '#888888')

    fig.add_trace(go.Scatter(
        x=df_c["Year"], y=df_c["Total Trade"],
        mode="lines+markers",
        name=country,
        line=dict(color=color, width=2.5),
        marker=dict(size=6),
        hovertemplate=f"<b>{country}</b><br>Year: %{{x}}<br>Total Trade: $%{{y:,.0f}}K USD<extra></extra>"
    ))

    # Switch year vertical line
    fig.add_vline(
        x=sy, line_dash="dash", line_color=color, line_width=1.5,
        annotation_text=f"{country[:3]} switches ({sy})",
        annotation_position="top",
        annotation_font_size=9
    )

fig.update_layout(
    title=dict(text="Taiwan Total Trade with Switcher Countries (2003–2024)",
               font=dict(size=18, color=ACCENT)),
    xaxis_title="Year",
    yaxis_title="Total Trade (USD $1,000)",
    legend_title="Country",
    hovermode="x unified",
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="Arial", size=13),
    xaxis=dict(showgrid=True, gridcolor="#eeeeee"),
    yaxis=dict(showgrid=True, gridcolor="#eeeeee"),
    height=500
)

fig.show()

In [4]:
# ── Chart 2: Taiwan Total Trade — Current Allies ──────────────────────
ally_data = tw_pivot[tw_pivot["Group"] == "Current Ally"].copy()

fig = px.line(
    ally_data,
    x="Year", y="Total Trade",
    color="Partner",
    markers=True,
    title="Taiwan Total Trade with Current Allies (2003–2024)",
    labels={"Total Trade": "Total Trade (USD $1,000)", "Partner": "Country"},
    hover_data={"Total Trade": ":,.0f", "Year": True}
)

fig.update_traces(
    hovertemplate="<b>%{fullData.name}</b><br>Year: %{x}<br>Total Trade: $%{y:,.0f}K USD<extra></extra>"
)
fig.update_layout(
    plot_bgcolor="white", paper_bgcolor="white",
    font=dict(family="Arial", size=13),
    xaxis=dict(showgrid=True, gridcolor="#eeeeee"),
    yaxis=dict(showgrid=True, gridcolor="#eeeeee"),
    title_font=dict(size=18, color=ACCENT),
    hovermode="x unified",
    height=500
)
fig.show()

## Exports vs. Imports

The animated bar charts below let you watch how each country's export and import rankings with Taiwan changed year by year from 2003 to 2024. Press **Play** to start the animation, or drag the year slider to jump to a specific year.

This is useful for seeing which countries consistently ranked highest in trade with Taiwan and whether switcher countries were rising or falling in the rankings before they switched.

In [6]:
# ── Chart 3: Animated Bar — Taiwan Exports Rankings ───────────────────
export_data = tw_focus[tw_focus["Flow"] == "Exports"].copy()
export_data = export_data[export_data["Partner"].isin(all_countries)]
export_data["Group"] = export_data["Partner"].apply(
    lambda x: "Switcher" if x in switch_years else "Current Ally"
)
export_rank = export_data.sort_values(["Year", "Value"], ascending=[True, False])

fig = px.bar(
    export_rank,
    x="Value", y="Partner",
    orientation="h",
    animation_frame="Year",
    color="Group",
    color_discrete_map={"Switcher": SWITCHER_COLOR, "Current Ally": ALLY_COLOR},
    range_x=[0, export_rank["Value"].max() * 1.05],
    hover_data={"Value": ":,.0f"},
    title="Taiwan Exports by Country (2003–2024)",
    labels={"Value": "Exports (USD $1,000)", "Partner": ""}
)
fig.update_layout(
    yaxis=dict(autorange="reversed"),
    plot_bgcolor="white", paper_bgcolor="white",
    font=dict(family="Arial", size=13),
    title_font=dict(size=18, color=ACCENT),
    height=520,
    legend_title="Country Type"
)
fig.show()

In [7]:
# ── Chart 4: Animated Bar — Taiwan Imports Rankings ───────────────────
import_data = tw_focus[tw_focus["Flow"] == "Imports"].copy()
import_data = import_data[import_data["Partner"].isin(all_countries)]
import_data["Group"] = import_data["Partner"].apply(
    lambda x: "Switcher" if x in switch_years else "Current Ally"
)
import_rank = import_data.sort_values(["Year", "Value"], ascending=[True, False])

fig = px.bar(
    import_rank,
    x="Value", y="Partner",
    orientation="h",
    animation_frame="Year",
    color="Group",
    color_discrete_map={"Switcher": SWITCHER_COLOR, "Current Ally": ALLY_COLOR},
    range_x=[0, import_rank["Value"].max() * 1.05],
    hover_data={"Value": ":,.0f"},
    title="Taiwan Imports by Country (2003–2024)",
    labels={"Value": "Imports (USD $1,000)", "Partner": ""}
)
fig.update_layout(
    yaxis=dict(autorange="reversed"),
    plot_bgcolor="white", paper_bgcolor="white",
    font=dict(family="Arial", size=13),
    title_font=dict(size=18, color=ACCENT),
    height=520,
    legend_title="Country Type"
)
fig.show()

## Before vs. After the Switch

This section compares the **average Taiwan trade volume** for switcher countries in the years **before** they switched recognition versus **after** they switched. For current allies, the full period average is shown.

If switching recognition to China meant a decline in Taiwan trade, we would expect the "After Switch" bars to be noticeably lower than the "Before Switch" bars.

In [8]:
# ── Chart 5: Before vs After Switch — Grouped Bar ─────────────────────
avg_trade = (
    tw_pivot.groupby(["Partner", "Period"])["Total Trade"]
    .mean().reset_index()
)

period_order = ["Before Switch", "After Switch", "Current Ally"]
color_map = {
    "Before Switch": ALLY_COLOR,
    "After Switch":  SWITCHER_COLOR,
    "Current Ally":  HIGHLIGHT
}

fig = px.bar(
    avg_trade,
    x="Partner", y="Total Trade",
    color="Period",
    barmode="group",
    category_orders={"Period": period_order},
    color_discrete_map=color_map,
    title="Average Taiwan Trade: Before vs. After Recognition Switch",
    labels={"Total Trade": "Avg Total Trade (USD $1,000)", "Partner": ""},
    hover_data={"Total Trade": ":,.0f"}
)
fig.update_layout(
    plot_bgcolor="white", paper_bgcolor="white",
    font=dict(family="Arial", size=13),
    title_font=dict(size=18, color=ACCENT),
    xaxis_tickangle=-30,
    legend_title="Period",
    hovermode="x unified",
    height=500
)
fig.show()

## Annual Trade Growth Rate

The chart below shows the **year-over-year percentage change** in total Taiwan trade for each country. Large positive spikes indicate years of rapid trade growth. Negative values indicate years where trade with Taiwan declined.

This is particularly revealing for switcher countries — we can see whether trade was already declining in the years before a country switched, or whether the switch itself caused the drop.

In [9]:
# ── Chart 6: Trade Growth Rate — Switchers vs Allies ──────────────────
growth_data = tw_pivot.dropna(subset=["Trade_Growth_Pct"]).copy()

fig = px.line(
    growth_data,
    x="Year", y="Trade_Growth_Pct",
    color="Partner",
    line_dash="Group",
    line_dash_map={"Switcher": "solid", "Current Ally": "dot"},
    markers=True,
    title="Annual Taiwan Trade Growth Rate by Country (%)",
    labels={"Trade_Growth_Pct": "Growth Rate (%)", "Partner": "Country"},
    hover_data={"Trade_Growth_Pct": ":.1f"}
)
fig.add_hline(y=0, line_dash="dash", line_color="gray", line_width=1)
fig.update_layout(
    plot_bgcolor="white", paper_bgcolor="white",
    font=dict(family="Arial", size=13),
    title_font=dict(size=18, color=ACCENT),
    hovermode="x unified",
    height=520
)
fig.show()

## Exports vs. Imports Side by Side

The faceted chart below separates **exports** (goods Taiwan sells to these countries) from **imports** (goods Taiwan buys from them), letting you compare both trade directions for each country simultaneously.

In [10]:
# ── Chart 7: Exports vs Imports faceted — Switchers ───────────────────
tw_long_sw = tw_focus[
    (tw_focus["Partner"].isin(switch_years.keys())) &
    (tw_focus["Flow"].isin(["Exports", "Imports"]))
].copy()

fig = px.line(
    tw_long_sw,
    x="Year", y="Value",
    color="Partner",
    facet_col="Flow",
    markers=True,
    title="Taiwan Exports vs. Imports — Switcher Countries (2003–2024)",
    labels={"Value": "Value (USD $1,000)", "Partner": "Country"},
)

# Add switch year lines per country
for country, sy in switch_years.items():
    fig.add_vline(x=sy, line_dash="dot", line_color=SWITCHER_COLORS.get(country, 'gray'),
                  line_width=1.2)

fig.update_layout(
    plot_bgcolor="white", paper_bgcolor="white",
    font=dict(family="Arial", size=12),
    title_font=dict(size=18, color=ACCENT),
    hovermode="x unified",
    height=480
)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

## Predictive Modeling

### Why Predicting Recognition Switches Did Not Work Well

The original goal of this project was to build a predictive model to answer: *Can trade volume, GDP, and regime type predict whether a country switches diplomatic recognition from Taiwan to China?*

Both a **Linear Regression** and a **Logistic Regression** were tested. The results showed very low R² values (0.015 for Model A, 0.040 for Model B) and no statistically significant predictors.

**The core reason this did not work is a data limitation:** only **11 out of 411** country-year observations in the dataset represent an actual switch. With only 11 positive cases, there is simply not enough variation in the outcome for any statistical model to detect a reliable signal. This is a known problem in political science when studying rare events — the data to train a model just does not exist yet because switching is so uncommon.

### A Better Question for This Data

Rather than predicting a rare binary event, a more appropriate question for the available data is:

> **Can we predict total Taiwan trade volume based on a country's GDP — and which current Taiwan allies show the strongest and most stable trade relationship?**

This is a standard **linear regression** problem that fits the data well, because total trade is a continuous variable with meaningful variation across all 24 countries and 22 years.

In [11]:
# ── Load merged panel for regression ──────────────────────────────────
panel_path = "/Users/jasminehernandez/Desktop/Taiwan Research Project/merged_datasets.csv"
panel = pd.read_csv(panel_path)

# Filter to current allies only, drop missing
allies_list = ["Guatemala", "Paraguay", "Belize", "Haiti", "Marshall Islands",
               "Palau", "Tuvalu", "Saint Kitts and Nevis",
               "Saint Lucia", "Saint Vincent and the Grenadines", "Eswatini"]

reg_df = panel[
    (panel["country"].isin(allies_list)) &
    (panel["year"].between(2003, 2023))
][[
    "country", "year",
    "taiwan_total_trade_usd_thousand",
    "gdp_current_usd"
]].dropna()

reg_df["gdp_billions"] = reg_df["gdp_current_usd"] / 1e9
reg_df["trade_millions"] = reg_df["taiwan_total_trade_usd_thousand"] / 1e3

print(f"Regression dataset: {len(reg_df)} observations across {reg_df['country'].nunique()} current allies")
reg_df.head()

Regression dataset: 231 observations across 11 current allies


,country,year,taiwan_total_trade_usd_thousand,gdp_current_usd,gdp_billions,trade_millions
0,Belize,2003,6691.16798,1.308280e+09,1.308280,6.691168
1,Belize,2004,8249.06965,1.400202e+09,1.400202,8.249070
2,Belize,2005,10871.49728,1.474298e+09,1.474298,10.871497
3,Belize,2006,6814.06107,1.590463e+09,1.590463,6.814061
4,Belize,2007,7883.74312,1.706190e+09,1.706190,7.883743


In [12]:
# ── Run Linear Regression: GDP → Taiwan Trade ─────────────────────────
X = reg_df[["gdp_billions"]].values
y = reg_df["trade_millions"].values

model = LinearRegression().fit(X, y)
y_pred = model.predict(X)
r2 = r2_score(y, y_pred)

print("=== Linear Regression: GDP → Taiwan Total Trade ===")
print(f"Intercept (β₀):  {model.intercept_:.4f}")
print(f"Coefficient (β₁): {model.coef_[0]:.4f}")
print(f"R²:               {r2:.4f}")
print()
print("Interpretation:")
print(f"For every $1 billion increase in GDP, Taiwan total trade increases")
print(f"by approximately ${model.coef_[0]:.2f} million USD on average.")
print(f"R² of {r2:.2f} means GDP explains {r2*100:.1f}% of the variation in Taiwan trade.")

=== Linear Regression: GDP → Taiwan Total Trade ===
Intercept (β₀):  6.6049
Coefficient (β₁): 3.1319
R²:               0.6577

Interpretation:
For every $1 billion increase in GDP, Taiwan total trade increases
by approximately $3.13 million USD on average.
R² of 0.66 means GDP explains 65.8% of the variation in Taiwan trade.


In [13]:
# ── Chart 8: Scatter — GDP vs Taiwan Trade with Regression Line ────────
reg_df["Predicted Trade"] = model.predict(X)

fig = px.scatter(
    reg_df,
    x="gdp_billions", y="trade_millions",
    color="country",
    hover_data={"year": True, "gdp_billions": ":.2f", "trade_millions": ":.1f"},
    title="GDP vs. Taiwan Total Trade — Current Allies (2003–2023)",
    labels={
        "gdp_billions": "GDP (USD Billions)",
        "trade_millions": "Taiwan Total Trade (USD Millions)",
        "country": "Country"
    }
)

# Add regression line
x_range = np.linspace(reg_df["gdp_billions"].min(), reg_df["gdp_billions"].max(), 100)
y_range = model.predict(x_range.reshape(-1,1))
fig.add_trace(go.Scatter(
    x=x_range, y=y_range,
    mode="lines",
    name=f"Regression Line (R²={r2:.2f})",
    line=dict(color=ACCENT, width=2.5, dash="dash")
))

fig.update_layout(
    plot_bgcolor="white", paper_bgcolor="white",
    font=dict(family="Arial", size=13),
    title_font=dict(size=18, color=ACCENT),
    height=520
)
fig.show()

In [14]:
# ── Chart 9: Average Taiwan Trade — Current Allies Ranked ─────────────
avg_ally = (
    reg_df.groupby("country")["trade_millions"]
    .mean().reset_index()
    .rename(columns={"trade_millions": "Avg Taiwan Trade (USD M)", "country": "Country"})
    .sort_values("Avg Taiwan Trade (USD M)", ascending=True)
)

fig = px.bar(
    avg_ally,
    x="Avg Taiwan Trade (USD M)", y="Country",
    orientation="h",
    color="Avg Taiwan Trade (USD M)",
    color_continuous_scale="Blues",
    title="Average Annual Taiwan Trade — Current Allies Ranked (2003–2023)",
    hover_data={"Avg Taiwan Trade (USD M)": ":.2f"}
)
fig.update_layout(
    plot_bgcolor="white", paper_bgcolor="white",
    font=dict(family="Arial", size=13),
    title_font=dict(size=18, color=ACCENT),
    coloraxis_showscale=False,
    height=460
)
fig.show()

print("\nTop current ally by average Taiwan trade:")
print(avg_ally.sort_values("Avg Taiwan Trade (USD M)", ascending=False).head(3).to_string(index=False))


Top current ally by average Taiwan trade:
         Country  Avg Taiwan Trade (USD M)
       Guatemala                192.927397
        Paraguay                 97.164160
Marshall Islands                 70.544259


## Summary and Key Takeaways

**1. Trade volumes declined for several switcher countries after switching recognition.**
Countries like Costa Rica and El Salvador show a noticeable drop in Taiwan trade after their switch years, while Honduras (2023) is too recent to draw firm conclusions.

**2. Paraguay is Taiwan's strongest current ally by trade volume.**
Paraguay consistently ranks first among current Taiwan allies in total bilateral trade, making it the most economically significant diplomatic partner Taiwan has among its remaining allies.

**3. GDP is a meaningful predictor of Taiwan trade among current allies.**
The linear regression shows that larger economies trade more with Taiwan, which is expected. This model performs much better than the recognition-switch prediction because trade volume has natural variation across countries and years.

**4. Predicting recognition switches is statistically limited by the rarity of the event.**
With only 11 switching events out of 411 observations, no regression model can reliably identify which countries are at risk of switching. Future research would need more historical cases or alternative modeling approaches designed for rare events.

*Dashboard created by Jasmine Hernandez | Capstone Project | April 2026*